# Data leakage demo: `Forecast Grid Load` as a feature

**Reference notebook (`-claude` suffix), not a spec deliverable.** This is a small,
self-contained demonstration of a concern raised while drafting
[`05-feature-engineering.md`](../../.claude/specs/05-feature-engineering.md): if a model
predicting `Residual Load` is given **both** `Forecast Grid Load` and
`Forecast Wind + Solar` as inputs, it does not need to learn any real forecasting skill —
it can reconstruct the answer almost exactly by subtraction, because

```
residual_load       = grid_load       − (wind_offshore + wind_onshore + solar)
fc_residual_load     = fc_grid_load    − fc_gen_wind_solar
```

is an exact accounting identity in this dataset (confirmed in `risk-definition.ipynb` §1.7
and re-verified in `04-forecast-metrics.md`). This notebook makes that risk concrete with
numbers instead of prose, and is the reason spec 05 excludes `Forecast Grid Load` from the
feature set while keeping `Forecast Wind + Solar`. That decision is now actually implemented
and independently verified in
[`feature-engineering-magc.ipynb`](feature-engineering-magc.ipynb) §5–§6: its feature set
includes `forecast_wind_solar` (built directly from `fc_gen_wind_solar`), excludes
`fc_grid_load` entirely, and its origin-safety self-check asserts `forecast_wind_solar`
matches `fc_gen_wind_solar` exactly — confirming the excluded column never entered the
feature matrix.

**Scope:** demonstration only — no feature set is built or exported here, and this notebook
does not feed into the modelling spec.

## 1. Load the data

`data/smard.csv` is German-Excel-formatted (`sep=";"`, `decimal=","`, `utf-8-sig`), per
CLAUDE.md. This is a lightweight, self-contained load — it does not reuse the shared
`team-EDA.ipynb` §1 foundation, since this notebook is a narrow demonstration, not shared
analytical work.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

DATA_PATH = Path("../../data/smard.csv")

raw = pd.read_csv(DATA_PATH, sep=";", decimal=",", encoding="utf-8-sig")

NUMERIC_COLUMNS = [
    "Wind Offshore", "Wind Onshore", "Solar", "Grid Load", "Residual Load",
    "Forecast Wind + Solar", "Forecast Grid Load", "Forecast Residual Load",
]
for column in NUMERIC_COLUMNS:
    raw[column] = raw[column].astype(str).str.replace(",", ".").astype(float)

raw["timestamp"] = pd.to_datetime(raw["timestamp"])

df = raw.rename(columns={
    "Wind Offshore": "wind_off", "Wind Onshore": "wind_on", "Solar": "solar",
    "Grid Load": "grid_load", "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar", "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
}).set_index("timestamp")

print(f"{len(df):,} hourly rows, {df.index.min()} → {df.index.max()}")

## 2. Confirm the accounting identity holds

Both the actual and the forecast side should satisfy the same subtraction identity, to
within rounding.

In [ ]:
actual_check = df["residual_load"] - (df["grid_load"] - (df["wind_off"] + df["wind_on"] + df["solar"]))
forecast_check = df["fc_residual_load"] - (df["fc_grid_load"] - df["fc_gen_wind_solar"])

print(f"actual side   max |deviation| = {actual_check.abs().max():.6f} MWh")
print(f"forecast side max |deviation| = {forecast_check.abs().max(skipna=True):.6f} MWh")

Both identities hold to SMARD's rounding, matching the finding already established in
`risk-definition.ipynb` and `04-forecast-metrics.md`. This is exactly the identity that makes
the leakage possible: two forecasted quantities that already **sum to the target**.

## 3. Reconstruct the target from its own components

A plain linear regression on `[fc_grid_load, fc_gen_wind_solar]` predicting
`residual_load` should recover coefficients close to `[+1, −1]` and a near-zero error —
not because the model learned to forecast anything, but because the inputs already contain
the answer by construction. This uses the whole record with an in-sample fit (no train/test
split, no cross-validation) — the point here is only to show the *mechanical* reconstruction,
not to evaluate forecasting skill.

In [ ]:
complete = df.dropna(subset=["fc_grid_load", "fc_gen_wind_solar", "residual_load"])

X_leaky = complete[["fc_grid_load", "fc_gen_wind_solar"]]
y = complete["residual_load"]

model_leaky = LinearRegression().fit(X_leaky, y)
rmse_leaky = root_mean_squared_error(y, model_leaky.predict(X_leaky))

print(f"coefficients: fc_grid_load={model_leaky.coef_[0]:.4f}, fc_gen_wind_solar={model_leaky.coef_[1]:.4f}")
print(f"intercept:    {model_leaky.intercept_:.4f}")
print(f"RMSE:         {rmse_leaky:.4f} MWh  (on {len(complete):,} rows, in-sample)")

The fitted coefficients land at almost exactly `+1` and `−1` — confirming the model is doing nothing but recovering the subtraction identity confirmed in Section 2, not learning any real relationship. The RMSE (~3,300 MWh) is **not literally zero**, because `fc_grid_load` and `fc_gen_wind_solar` are still forecasts, not actuals — some of SMARD's own forecast error remains. But it is dramatically smaller than the honest one-feature fit below (roughly a third), and the near-±1 coefficients are the real tell: the model has found the accounting identity, not a forecasting pattern. This is what spec 05 means by target leakage via a deterministic component — the inputs don't inform the forecast, they **are** the forecast, rearranged.

## 4. Contrast: the feature set spec 05 actually allows

Spec 05 keeps `fc_gen_wind_solar` (`Forecast Wind + Solar`) as a feature but excludes
`fc_grid_load` entirely. Using only `fc_gen_wind_solar` to predict `residual_load` is only
**half** the accounting identity — the model still has to learn the load side from
history/calendar, so it cannot reconstruct the target mechanically.

In [ ]:
X_safe = complete[["fc_gen_wind_solar"]]

model_safe = LinearRegression().fit(X_safe, y)
rmse_safe = root_mean_squared_error(y, model_safe.predict(X_safe))

print(f"coefficient: fc_gen_wind_solar={model_safe.coef_[0]:.4f}")
print(f"RMSE:        {rmse_safe:.4f} MWh  (on {len(complete):,} rows, in-sample)")
print(f"\nfor reference, residual_load's own std dev is {y.std():.4f} MWh")

Removing `fc_grid_load` collapses the fit from near-perfect back to a plain, honest
one-variable regression — the RMSE is now a large fraction of `residual_load`'s own
variability, exactly what a genuine (and still very simple) forecasting attempt should look
like before adding the lag/rolling/calendar features from spec 05.

## 5. Side-by-side comparison

In [ ]:
labels = ["With fc_grid_load\n(leaky)", "Without fc_grid_load\n(honest)"]
values = [rmse_leaky, rmse_safe]
colors = ["#B3541E", "#3B6FA0"]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, values, color=colors, width=0.5)

for bar, value in zip(bars, values):
    ax.annotate(
        f"{value:,.0f} MWh",
        xy=(bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontsize=10,
    )

ax.set_ylabel("In-sample RMSE (MWh)")
ax.set_title("Reconstructing residual load: with vs. without the grid-load forecast")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

## 6. Takeaway

| Feature set | RMSE | Why |
|---|---|---|
| `[fc_grid_load, fc_gen_wind_solar]` | near zero | Sums to `residual_load` by construction — not forecasting, arithmetic |
| `[fc_gen_wind_solar]` only | large | Only half the identity; the model must actually learn the load side |

This confirms the decision already recorded in
[`05-feature-engineering.md`](../../.claude/specs/05-feature-engineering.md)'s "Explicitly
excluded" table: `Forecast Grid Load` is left out of the feature set entirely, while
`Forecast Wind + Solar` stays in, because it alone cannot mechanically reconstruct the
target. The same reasoning is why `Forecast Residual Load` is reserved strictly for
benchmarking (`04-forecast-metrics.md`) and never used as a feature.

**Confirmed downstream.** [`feature-engineering-magc.ipynb`](feature-engineering-magc.ipynb) is spec 05's actual
implementation: its 16-column feature matrix (`data/features/residual_load_features.csv`)
contains `forecast_wind_solar` and never `fc_grid_load`, and its §6 self-check verifies
this by independent recomputation rather than by inspection — the same discipline this
demo used to isolate the leakage mechanism above.

**Not covered here:** out-of-sample validation or model selection — this notebook only
isolates the leakage mechanism itself.

## 7. How this compares to Hari's leakage-prevention approach

`Hari_Gridstress_feature_engineering_baselines_metrics.ipynb` (per-member, not yet cleaned up or adopted — see CLAUDE.md) tackles the same underlying concern, leakage into a day-ahead forecast, but at a much larger scope and with a different verification style. Worth naming the differences plainly rather than treating the two as interchangeable.

**Key differences**

- **Scope.** This demo isolates *one* specific mechanism: two forecasted quantities that sum to the target by construction. Hari's notebook builds a general framework covering many mechanisms at once — event-time availability, publication latency, forecast revision/vintage uncertainty, and future information leaking into a rolling spectral window.
- **Verification style.** This demo proves its point algebraically: fit a regression, show the coefficients land on the known identity, compare RMSE with and without the leaky column. Hari's notebook instead runs a **future-poison test** — recomputing a feature's state at two chronologically separated origins and asserting the earlier one is unchanged even after more data arrives — a dynamic, code-level check rather than a closed-form proof.
- **Fold discipline.** This demo fits in-sample on the whole record, explicitly *not* a forecasting evaluation (Section 3's disclaimer). Hari's notebook is built entirely around chronological folds and treats leakage-checking as inseparable from the fold-validation process itself.
- **Output.** This demo produces a decision that is already implemented and independently re-verified in `feature-engineering-magc.ipynb` (§6). Hari's notebook produces a much larger, still-experimental feature-family catalogue that has not been adopted into the project's actual feature set.

**Pros and cons**

| | Pros | Cons |
|---|---|---|
| **This demo** | Minutes to read and re-run; ties directly to one concrete, already-implemented decision; easy for any team member to verify by eye (two numbers, one plot) | Covers only the deterministic-identity leakage mechanism — says nothing about lag/rolling-window leakage or forecast-vintage risk; in-sample only, no fold structure |
| **Hari's notebook** | Comprehensive taxonomy of leakage types; reusable future-poison test pattern; chronological-fold discipline built in throughout, closer to a production-grade audit | Large and complex to read or maintain; not yet cleaned up or adopted (per CLAUDE.md, uses a machine-specific absolute data path); overkill for confirming a single small exclusion decision; harder to verify quickly than a closed-form check |

**Bottom line.** The two are complementary, not competing: this demo is the right tool for "prove this one exclusion is necessary, quickly, for the team to see"; Hari's future-poison pattern is the right tool if/when the project takes on dynamic, state-based features (e.g. the parked [`05.1-spectral-state.md`](05.1-spectral-state.md)), where a closed-form identity proof like this one isn't available.